# Plataforma IoT con Databricks Data Intelligence Platform - Ingesta de datos de sensores industriales en tiempo real para Mantenimiento Prescriptivo

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_0.png" style="float: left; margin-right: 30px" width="600px" />

<br/>

## ¿Qué es la Databricks Data Intelligence Platform para IoT y Manufactura?
La Databricks Data Intelligence Platform para Manufactura desbloquea todo el valor de los datos industriales, habilitando redes inteligentes, mejores experiencias de cliente, productos más inteligentes y negocios sostenibles. Potencia a los equipos de datos con escalabilidad sin igual, insights en tiempo real y capacidades innovadoras en todos los tipos y fuentes de datos. Los fabricantes se benefician de costos reducidos, mayor productividad, mejor respuesta al cliente e innovación acelerada. La plataforma integra fuentes de datos diversas con procesamiento de IA de primer nivel y ofrece Solution Accelerators específicos para manufactura y partners para una toma de decisiones en tiempo real.
<img src="https://github.com/Datastohne/demo/blob/main/Intelligence%20Engine.png?raw=true " style="float: left; margin-right: 30px" width="200px" />

**Inteligente**
Databricks combina IA generativa con los beneficios de un lakehouse para impulsar un Motor de Data Intelligence que entiende la semántica única de tus datos. Esto permite que la plataforma optimice automáticamente el rendimiento y gestione la infraestructura de forma alineada a tu negocio.

<img src="https://github.com/Datastohne/demo/blob/main/24840.png?raw=true " style="float: right; margin-left: 30px" width="200px" />

**Simple** El lenguaje natural simplifica sustancialmente la experiencia de uso. El Motor de Data Intelligence entiende el lenguaje de tu organización, por lo que buscar y descubrir nuevos datos es tan fácil como hacer una pregunta. Además, el desarrollo de datos y aplicaciones se acelera con asistencia en lenguaje natural para escribir código, remediar errores y encontrar respuestas.

<img src="https://github.com/Datastohne/demo/blob/main/24841.png?raw=true " style="float: left; margin-right: 30px" width="200px" />

**Privada** Las aplicaciones de datos e IA requieren gobernanza y seguridad sólidas, especialmente con la IA generativa. Databricks ofrece una solución integral de MLOps y desarrollo de IA basada en un enfoque unificado de gobernanza y seguridad. Puedes llevar a cabo tus iniciativas de IA —desde APIs como OpenAI hasta modelos a medida— sin comprometer la privacidad ni el control de la propiedad intelectual.
 
<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=984752964297111&notebook=%2F00-IOT-wind-turbine-introduction-DI-platform&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F00-IOT-wind-turbine-introduction-DI-platform&version=1&user_hash=53e7df68e5fee236d97fc15226aeeed74331d6f7c2f836f9b915aecc5654a25a">

## Mantenimiento prescriptivo de turbinas eólicas con Databricks Data Intelligence Platform: llevando IA generativa al mantenimiento predictivo

Poder recopilar y centralizar información de equipos industriales en tiempo real es crítico en energía. Cuando una turbina eólica se detiene, deja de generar energía, lo que deriva en mala experiencia de cliente y pérdidas. Los datos son la clave para habilitar capacidades como optimización de energía, detección de anomalías y/o mantenimiento predictivo. El auge de la IA generativa permite revolucionar el mantenimiento: no solo prediciendo cuándo fallará un equipo, sino también generando acciones prescriptivas para prevenir fallas y optimizar el rendimiento. Esto impulsa el paso de mantenimiento predictivo a prescriptivo. <br/>

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/prescriptive_maintenance.png" width="700px" style="float:right; margin-left: 20px"/>

Ejemplos de mantenimiento prescriptivo:

- Analizar en tiempo real datos IoT de sensores del equipo
- Predecir fallas mecánicas en un ducto de energía
- Diagnosticar causas raíz de fallas previstas y generar acciones prescriptivas
- Detectar comportamientos anómalos en una línea de producción
- Optimizar el abastecimiento y staging de repuestos para mantenimientos programados

### Qué vamos a construir

En esta demo construiremos una plataforma IoT end-to-end para recolectar datos en tiempo real desde múltiples fuentes.

Crearemos un modelo predictivo para anticipar fallas en turbinas eólicas y lo usaremos para generar órdenes de trabajo, reduciendo inactividad y aumentando el OEE.

Además, desarrollaremos un dashboard para que el equipo de Mantenimiento monitoree turbinas, identifique riesgos y revise órdenes de trabajo, asegurando nuestras metas de productividad.

A alto nivel, este es el flujo que implementaremos:

<div style="text-align: center;">
    <img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/team_flow_overview.png" width="1000px">
</div>

1. Ingerir y crear nuestra base IoT y tablas, fácilmente consultables con SQL.
2. Asegurar datos y otorgar acceso de lectura a Analistas y Científicos de Datos.
3. Ejecutar consultas BI para analizar fallas existentes.
4. Construir un modelo de ML para monitorear el parque eólico y disparar mantenimiento predictivo.
5. Generar órdenes de trabajo para ingenieros de campo usando IA generativa.

Predecir qué turbina podría fallar es solo el primer paso para mejorar la eficiencia del parque. Con un modelo que anticipe mantenimiento, podemos adaptar dinámicamente el stock de repuestos, generar órdenes de trabajo y hasta despachar automáticamente equipos con el equipamiento adecuado.

### Nuestro dataset

Para simplificar, asumimos que un sistema externo envía periódicamente datos a nuestro blob storage (S3/ADLS/GCS):

- Datos de turbina *(ubicación, modelo, identificador, etc.)*
- Sensores de turbina, cada segundo *(energía producida, vibración; típicamente en streaming)*
- Estado de turbina en el tiempo, etiquetado por analistas, y reportes históricos de mantenimiento *(histórico para entrenar el modelo y para indexar en base vectorial)*

*Nota: técnicamente los datos pueden venir de cualquier fuente. Databricks ingiere desde Salesforce, Fivetran, colas como Kafka, blob storage, bases SQL/NoSQL, etc.*

Veamos cómo usar estos datos en la Data Intelligence Platform para analizar sensores, activar mantenimiento predictivo y generar órdenes de trabajo.

## 1/ Ingesta y preparación de datos (Data Engineering)

<img style="float: left; margin-right: 20px" width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_1.png" />


<br/>
<div style="padding-left: 420px">
Nuestro primer paso es ingerir y limpiar los datos en bruto para que el equipo de Analistas pueda empezar a analizarlos.


<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo.png" style="float: right; margin-top: 20px" width="200px">

### Delta Lake

Todas las tablas que creemos en el Lakehouse se almacenarán como tablas Delta Lake. [Delta Lake](https://delta.io) es un framework de almacenamiento abierto por confiabilidad y rendimiento. <br/>
Proporciona funcionalidades como *(Transacciones ACID, DELETE/UPDATE/MERGE, clonación sin copia, Change Data Capture...)* <br />
Para más detalles sobre Delta Lake, ejecuta `dbdemos.install('delta-lake')`

### Simplificar la ingesta con Spark Declarative Pipelines

Databricks simplifica la ingesta y transformación de datos con Spark Declarative Pipelines permitiendo a usuarios de SQL crear pipelines avanzados en batch o streaming. También simplifica el despliegue, las pruebas y el seguimiento de la calidad de datos, reduciendo la complejidad operativa para que te enfoques en las necesidades del negocio.<br/>

Abre el
  <a dbdemos-pipeline-id="sdp-sql" href="#joblist/pipelines/4d2b04f6-270a-49da-9980-96f393f6ef8c" target="_blank">pipeline de Spark Declarative Pipelines</a> o el [notebook SQL]($./01-Data-ingestion/01.1-SDP-SQL/01.1-SDP-Wind-Turbine-SQL) *(Alternativas:  [Spark Declarative Pipelines en Python]($./01-Data-ingestion/01.2-SDP-python/01.1-SDP-Wind-Turbine-Python) - [versión Delta+Spark]($./01-Data-ingestion/plain-spark-delta-pipeline/01.5-Delta-pipeline-spark-iot-turbine))*. <br>
  Para más detalles sobre Spark Declarative Pipelines: `dbdemos.install('pipeline-bike')` o `dbdemos.install('declarative-pipeline-cdc')`} (json)``` remodeled code to match expected JSON schema.  Hope it helps.  Thanks.  !  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!  !!

## 2/ Aseguramiento de datos y gobernanza (Unity Catalog)

<img width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_2.png"  style="float: left; margin-right: 10px"/>

<br/><br/><br/>
<div style="padding-left: 420px">
  Ahora que creamos las primeras tablas, debemos otorgar acceso de LECTURA al equipo de Analistas para que puedan empezar a analizar la información de fallas.
  
  Veamos cómo Unity Catalog provee seguridad y gobernanza sobre nuestros activos de datos e incluye lineage y audit logs.
  
  Ten en cuenta que Unity Catalog integra Delta Sharing, un protocolo abierto para compartir datos con organizaciones externas, independientemente de su stack o nube. Más detalles:  `dbdemos.install('delta-sharing-airlines')`
 </div>

   Abre el [notebook de Unity Catalog]($./02-Data-governance/02-UC-data-governance-security-iot-turbine) para ver cómo configurar ACL y explorar lineage con el Data Explorer.
  

## 3/ Analizando fallas (BI / Data Warehousing / SQL) 

<img width="300px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-1.png"  style="float: right; margin: 100px 0px 10px;"/>

<img width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_2.png"  style="float: left; margin-right: 10px"/>
 
<br><br><br>
Nuestros datasets ya están correctamente ingeridos, seguros, con alta calidad y fácilmente descubribles dentro de la organización.

Los Analistas de Datos están listos para ejecutar consultas BI interactivas de baja latencia y alto rendimiento. Pueden crear un clúster nuevo, usar un clúster compartido, o para tiempos de respuesta aún más rápidos, usar Databricks Serverless SQL Warehouses que permiten arranque/detención instantáneos.

¡Veamos cómo se hace Data Warehousing en Databricks! Revisaremos dashboards incorporados: la plataforma cubre de la ingesta al análisis e integra herramientas BI populares como PowerBI, Tableau y más.

Abre el [notebook de Data Warehousing]($./03-BI-data-warehousing/03-BI-Datawarehousing-iot-turbine) para empezar a ejecutar consultas BI o abre directamente el <a dbdemos-dashboard-id="turbine-analysis" href="/sql/dashboardsv3/01f0b6a83d751bcb973f7564d17676fd" target="_blank">dashboard AI/BI de análisis de turbinas</a>

## 4/ Predecir fallas con Data Science y AutoML

<img width="500px" style="float: left; margin-right: 10px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform3.png" />

<br><br><br>
Analizar los datos históricos nos dio muchos insights para impulsar el negocio. Ahora entendemos mejor el impacto del downtime y qué turbinas están caídas en nuestro dashboard casi en tiempo real.

Sin embargo, saber qué turbinas fallaron no es suficiente. Debemos ir más allá y construir un modelo predictivo para detectar fallas potenciales antes de que ocurran, aumentando uptime y minimizando costos.

Aquí es donde brilla el Lakehouse. En la misma plataforma cualquiera puede crear un modelo de ML para predecir fallas, ya sea con desarrollo tradicional o con nuestra solución low‑code AutoML.

Veamos cómo entrenar un modelo de ML con 1 clic con el [04.1-automl-iot-turbine-predictive-maintenance]($./04-Data-Science-ML/04.1-automl-iot-turbine-predictive-maintenance)

## 5/ Generar órdenes de trabajo de mantenimiento con IA generativa

<img width="500px" style="float: left; margin-right: 10px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_4.png" />

<br><br><br>

El auge de la IA generativa habilita el paso de modelos de mantenimiento predictivo a prescriptivo. Al pasar de modelos de ML a sistemas de agentes, podemos aprovechar el modelo predictivo como uno de los componentes del sistema de IA. Esto abre oportunidades para automatización y eficiencia que aumentan el uptime y minimizan costos.

Databricks ofrece un conjunto de herramientas para construir, desplegar y evaluar agentes de IA de calidad productiva como aplicaciones de Retrieval Augmented Generation (RAG), incluyendo base vectorial, endpoints de model serving, gobernanza, monitoreo y evaluación. 

_Disclaimer: si tu organización aún no permite usar Databricks Vector Search y/o Model Serving, puedes omitir esta sección._

Creemos nuestro primer sistema de agentes con el [05.1-ai-tools-iot-turbine-prescriptive-maintenance]($./05-Generative-AI/05.1-ai-tools-iot-turbine-prescriptive-maintenance)


## Automatizar acciones para reducir cortes de turbina basados en predicciones


<img style="float: right" width="400px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-2.png">

Ahora tenemos un pipeline end‑to‑end que analiza datos de sensores, detecta fallas potenciales y genera acciones prescriptivas basadas en reportes históricos de mantenimiento para prevenir fallas antes de que ocurran. Con esto, podemos disparar acciones de seguimiento para reducir cortes, por ejemplo:

- Programar mantenimiento según disponibilidad de equipos y gravedad de la falla
- Preparar partes e insumos para operaciones de mantenimiento predictivo, manteniendo bajo stock
- Medir la eficiencia del modelo de mantenimiento predictivo y su ROI

*Nota: Estas acciones están fuera del alcance de esta demo y solo aprovechan los resultados del modelo de mantenimiento predictivo.*


Abre el <a dbdemos-dashboard-id="turbine-predictive" href="/sql/dashboardsv3/01f0b6a83d751bcb973f7564d17676fd">dashboard AI/BI de mantenimiento prescriptivo</a> para una vista completa del parque eólico, incluyendo turbinas potencialmente defectuosas, órdenes de trabajo y acciones de remediación.

## 6/ Desplegar y orquestar el flujo completo

<img style="float: left; margin-right: 10px" width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/di_platform_5.png" />

<br><br><br>
Aunque nuestro pipeline está casi completo, falta un último paso: orquestar el flujo en producción.

Con Databricks Lakehouse no necesitas un orquestador externo. Workflows simplifica tus jobs con alertas avanzadas, monitoreo, branching, etc.

Abre el [notebook de orquestación y workflows]($./06-Workflow-orchestration/06-Workflow-orchestration-iot-turbine) para programar nuestro pipeline (ingesta de datos, reentrenamiento del modelo, etc.)


## Conclusión

Demostramos cómo implementar un pipeline end‑to‑end con el Lakehouse usando una plataforma unificada y segura. Vimos:

- Ingesta de datos
- Análisis de datos / DW / BI 
- Data Science / ML
- IA generativa
- Workflow y Orquestación

Como resultado, el equipo de negocio puede no solo entender mejor las fallas sino también anticiparlas y permitir que mantenimiento actúe en consecuencia.

*Esto fue solo una introducción a la plataforma Databricks. Para más detalles, contacta a tu equipo de cuenta y explora más demos con `dbdemos.list()`!*